# Introdução ao MPI no Google Colab

**Computação Paralela e Distribuída — Prof. Paulo Bressan**

> **Observação importante:** o Colab roda em uma única máquina (um único nó). Por isso, este notebook simula múltiplos processos MPI *na mesma máquina*, o que é suficiente para aprender a sintaxe e os conceitos de comunicação entre processos (send/recv, broadcast, reduce), mas não representa um cluster distribuído real com múltiplas máquinas físicas.

Este notebook cobre:
1. Instalação do MPI e da biblioteca `mpi4py`
2. "Hello World" em MPI (identificação de processos)
3. Comunicação ponto a ponto (`send` / `recv`)
4. Broadcast (`bcast`)
5. Redução paralela (`reduce`) — soma distribuída

## 1. Instalação do MPI e do mpi4py

Execute esta célula primeiro. Ela instala o MPICH (implementação do padrão MPI) e a biblioteca Python `mpi4py`.

In [1]:
!apt-get install -y mpich > /dev/null 2>&1
!pip install mpi4py > /dev/null 2>&1
print("Instalação concluída.")

Instalação concluída.


**Sobre o `--oversubscribe`:** as VMs do Colab geralmente têm apenas 2 núcleos de CPU. Por padrão, o Open MPI não permite lançar mais processos do que núcleos disponíveis, para evitar sobrecarga. Como aqui estamos simulando processos MPI (não fazendo paralelismo real de hardware), usamos a flag `--oversubscribe` nas próximas células para liberar esse limite. Você pode conferir quantos núcleos o Colab lhe deu com o comando abaixo.

In [ ]:
!nproc

## 2. Hello World em MPI

Cada processo MPI imprime seu próprio rank (identificador) e o total de processos (`size`).

O comando `%%writefile` salva o conteúdo da célula como um arquivo `.py`, que depois é executado com `mpirun`.

In [ ]:
%%writefile hello_mpi.py
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()   # identificador do processo (0, 1, 2, ...)
size = comm.Get_size()   # número total de processos

print(f"Olá do processo {rank} de {size}!")

In [ ]:
# --allow-run-as-root é necessário porque o Colab roda como root
!mpirun --allow-run-as-root --oversubscribe -np 4 python hello_mpi.py

### Versão em C

O mesmo exemplo, agora na linguagem C — a forma mais comum de se programar com MPI na prática (inclusive fora do Python). Repare que a lógica é idêntica à versão em `mpi4py`: cada processo descobre seu `rank` e o `size` total, e imprime uma mensagem.

In [ ]:
%%writefile hello_mpi.c
/* hello_mpi.c */
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);               // inicializa o ambiente MPI

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank); // id do processo
    MPI_Comm_size(MPI_COMM_WORLD, &size); // número total de processos

    char name[MPI_MAX_PROCESSOR_NAME];
    int len;
    MPI_Get_processor_name(name, &len);   // nome do nó (opcional)

    printf("Olá do processo %d de %d (host: %s)\n", rank, size, name);

    MPI_Finalize();                       // finaliza o ambiente MPI
    return 0;
}

**Compilar e Executar (linha de comando do Linux):**

```bash
mpicc -o hello_mpi hello_mpi.c
mpirun -np 4 ./hello_mpi
```

In [ ]:
!mpicc -o hello_mpi hello_mpi.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./hello_mpi

**Executando em múltiplas máquinas com um hostfile**

Em um cluster real (várias máquinas físicas), o MPI precisa saber quais hosts usar e quantos processos ("slots") cada um pode rodar. Isso é feito com um *hostfile*.

Arquivo `hostfile.txt`:
```
localhost slots=8
```

Execução com hostfile:
```bash
mpirun --hostfile hostfile.txt -np 10 ./meu_programa
```

> No Colab isso não se aplica de verdade (só existe uma máquina), mas é assim que se distribuem processos entre nós reais em um cluster.

**Descrição de cada função MPI usada:**

| Função | O que faz |
|---|---|
| `MPI_Init(&argc, &argv)` | Inicializa o ambiente MPI. Deve ser a primeira chamada MPI do programa. |
| `MPI_Comm_rank(MPI_COMM_WORLD, &rank)` | Obtém o identificador (rank) do processo atual dentro do comunicador `MPI_COMM_WORLD` — um número de 0 a `size-1`. |
| `MPI_Comm_size(MPI_COMM_WORLD, &size)` | Obtém o número total de processos que estão participando da execução. |
| `MPI_Get_processor_name(name, &len)` | Obtém o nome do nó/host físico em que o processo está sendo executado (útil para identificar em qual máquina do cluster cada processo roda). |
| `MPI_Finalize()` | Encerra o ambiente MPI e libera os recursos associados. Deve ser a última chamada MPI do programa. |

## 3. Comunicação ponto a ponto (send / recv)

O processo de rank 0 envia uma mensagem para o processo de rank 1.

In [ ]:
%%writefile sendrecv_mpi.py
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

if rank == 0:
    dado = {"mensagem": "Olá do processo 0!", "valor": 42}
    comm.send(dado, dest=1, tag=11)
    print(f"[Processo 0] Enviou: {dado}")
elif rank == 1:
    dado_recebido = comm.recv(source=0, tag=11)
    print(f"[Processo 1] Recebeu: {dado_recebido}")
else:
    print(f"[Processo {rank}] Não participa desta troca de mensagens.")

In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python sendrecv_mpi.py

### Versão em C

O mesmo exemplo, agora em C: o processo 0 envia uma mensagem de texto e um valor inteiro para o processo 1; os demais processos apenas informam que não participam da troca.

In [ ]:
%%writefile sendrecv_mpi.c
/* sendrecv_mpi.c */
#include <mpi.h>
#include <stdio.h>
#include <string.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);

    int rank;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    if (rank == 0) {
        char mensagem[50] = "Ola do processo 0!";
        int valor = 42;

        // envia a mensagem de texto para o processo 1, com tag 11
        MPI_Send(mensagem, strlen(mensagem) + 1, MPI_CHAR, 1, 11, MPI_COMM_WORLD);
        // envia o valor inteiro para o processo 1, com a mesma tag 11
        MPI_Send(&valor, 1, MPI_INT, 1, 11, MPI_COMM_WORLD);

        printf("[Processo 0] Enviou: mensagem='%s', valor=%d\n", mensagem, valor);
    } else if (rank == 1) {
        char mensagem_recebida[50];
        int valor_recebido;

        // recebe a mensagem de texto enviada pelo processo 0
        MPI_Recv(mensagem_recebida, 50, MPI_CHAR, 0, 11, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        // recebe o valor inteiro enviado pelo processo 0
        MPI_Recv(&valor_recebido, 1, MPI_INT, 0, 11, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        printf("[Processo 1] Recebeu: mensagem='%s', valor=%d\n", mensagem_recebida, valor_recebido);
    } else {
        printf("[Processo %d] Nao participa desta troca de mensagens.\n", rank);
    }

    MPI_Finalize();
    return 0;
}

**Compilar e Executar:**

```bash
mpicc -o sendrecv_mpi sendrecv_mpi.c
mpirun -np 4 ./sendrecv_mpi
```

In [ ]:
!mpicc -o sendrecv_mpi sendrecv_mpi.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./sendrecv_mpi

### Outro exemplo em C

A seguir, um exemplo diferente de comunicação ponto a ponto em C: em vez de apenas um par de processos trocando mensagens, aqui **todos** os processos com rank diferente de 0 enviam uma saudação para o processo 0, que recebe e imprime todas elas.

In [ ]:
%%writefile greetings.c
#include <stdio.h>
#include <string.h>  /* For strlen             */
#include <mpi.h>     /* For MPI functions, etc */

const int MAX_STRING = 100;

int main(void) {
   char       greeting[MAX_STRING];  /* String storing message */
   int        comm_sz;               /* Number of processes    */
   int        my_rank;               /* My process rank        */

   /* Start up MPI */
   MPI_Init(NULL, NULL);                                   // inicializa o ambiente MPI

   /* Get the number of processes */
   MPI_Comm_size(MPI_COMM_WORLD, &comm_sz);                // número total de processos

   /* Get my rank among all the processes */
   MPI_Comm_rank(MPI_COMM_WORLD, &my_rank);                // id do processo atual

   if (my_rank != 0) {
      /* Create message */
      sprintf(greeting, "Greetings from process %d of %d!",  my_rank, comm_sz);
      /* Send message to process 0 */
      MPI_Send(greeting, strlen(greeting) + 1, MPI_CHAR, 0, 0, MPI_COMM_WORLD);
      // envia a string 'greeting' (com terminador incluso) para o processo de rank 0, com tag 0
   } else {
      /* Print my message */
      printf("Greetings from process %d of %d!\n", my_rank, comm_sz);
      for ( int q = 1 ; q < comm_sz ; q++) {
         /* Receive comm_smessage from process q */
         MPI_Recv(greeting, MAX_STRING, MPI_CHAR, q, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
         // recebe uma mensagem especificamente do processo de rank q, com tag 0,
         // bloqueando até que a mensagem chegue; MPI_STATUS_IGNORE descarta os metadados da recepção
         /* Print message from process q */
         printf("%s\n", greeting);
      }
   }

   /* Shut down MPI */
   MPI_Finalize();                                          // finaliza o ambiente MPI

   return 0;
}  /* main */

**Compilar e Executar:**

```bash
mpicc -o greetings greetings.c
mpirun -np 4 ./greetings
```

In [ ]:
!mpicc -o greetings greetings.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./greetings

**Descrição das funções `MPI_Send` e `MPI_Recv`:**

| Função | O que faz |
|---|---|
| `MPI_Send(buf, count, datatype, dest, tag, comm)` | Envia uma mensagem de forma bloqueante. `buf` é o endereço dos dados a enviar, `count` é o número de elementos, `datatype` é o tipo MPI dos dados (ex.: `MPI_CHAR`, `MPI_INT`), `dest` é o rank do processo destino, `tag` é um rótulo inteiro para identificar a mensagem, e `comm` é o comunicador (normalmente `MPI_COMM_WORLD`). |
| `MPI_Recv(buf, count, datatype, source, tag, comm, status)` | Recebe uma mensagem de forma bloqueante, aguardando até que ela chegue. `buf` é onde os dados recebidos serão armazenados, `count` é o tamanho máximo do buffer, `source` é o rank do processo do qual se espera a mensagem, `tag` deve casar com o tag usado no envio, e `status` (aqui `MPI_STATUS_IGNORE`) armazenaria metadados sobre a mensagem recebida (como o tamanho real e a origem), caso fossem necessários. |

## 4. Broadcast (bcast)

O processo 0 gera um valor e o transmite (broadcast) para todos os outros processos.

In [ ]:
%%writefile bcast_mpi.py
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

if rank == 0:
    dado = {"parametro": "taxa_aprendizado", "valor": 0.01}
else:
    dado = None

dado = comm.bcast(dado, root=0)

print(f"[Processo {rank}] Recebeu via broadcast: {dado}")

In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python bcast_mpi.py

### Versão em C

O mesmo exemplo, agora em C: o processo 0 (raiz) define um nome de parâmetro e um valor, e os transmite para todos os demais processos com `MPI_Bcast`.

In [ ]:
%%writefile bcast_mpi.c
/* bcast_mpi.c */
#include <mpi.h>
#include <stdio.h>
#include <string.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);

    int rank;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    char parametro[30];
    double valor;

    if (rank == 0) {
        strcpy(parametro, "taxa_aprendizado");
        valor = 0.01;
    }

    // transmite a string 'parametro' do processo 0 (raiz) para todos os processos
    MPI_Bcast(parametro, 30, MPI_CHAR, 0, MPI_COMM_WORLD);
    // transmite o valor 'valor' (double) do processo 0 (raiz) para todos os processos
    MPI_Bcast(&valor, 1, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    printf("[Processo %d] Recebeu via broadcast: parametro='%s', valor=%f\n", rank, parametro, valor);

    MPI_Finalize();
    return 0;
}

**Compilar e Executar:**

```bash
mpicc -o bcast_mpi bcast_mpi.c
mpirun -np 4 ./bcast_mpi
```

In [ ]:
!mpicc -o bcast_mpi bcast_mpi.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./bcast_mpi

**Descrição da função `MPI_Bcast`:**

| Função | O que faz |
|---|---|
| `MPI_Bcast(buf, count, datatype, root, comm)` | Transmite (broadcast) os dados presentes em `buf` no processo `root` para todos os demais processos do comunicador `comm`. Diferente de `MPI_Send`/`MPI_Recv`, é uma operação **coletiva**: todos os processos (inclusive a raiz) devem chamar `MPI_Bcast` — no processo raiz, `buf` é a origem dos dados; nos demais, `buf` é onde os dados recebidos serão armazenados. |

## Exercícios propostos

**Exercício 1.** Modifique o programa `greetings.c` (seção 3) para que, em vez de cada processo enviar apenas uma saudação de texto, cada processo `!= 0` envie um número inteiro correspondente ao seu próprio `rank` elevado ao quadrado (ex.: o processo 3 envia 9). O processo 0 deve receber os valores de todos os outros processos e imprimir a soma total. *(Dica: troque `MPI_CHAR` por `MPI_INT` nas chamadas de `MPI_Send`/`MPI_Recv`.)*

**Exercício 2.** Usando `MPI_Send` e `MPI_Recv`, escreva um programa em C no qual o processo 0 envie um vetor de 10 inteiros para o processo 1, que deve recebê-lo, calcular a soma dos elementos e enviar o resultado de volta para o processo 0 usando outro par `MPI_Send`/`MPI_Recv`. O processo 0 deve imprimir o resultado final recebido. Execute com `-np 2`.